In [0]:
%sql
CREATE TABLE IF NOT EXISTS zeb_labs.default.genie_cost_daily (
  query_date              DATE,
  executed_by             STRING,
  genie_space_id          STRING,
  warehouse_id            STRING,
  warehouse_size          STRING,
  dbu_per_hour            INT,
  execution_status        STRING,
  query_count             INT,
  total_duration_ms       BIGINT,
  total_estimated_dbus    DOUBLE,
  price_per_dbu           DOUBLE,
  estimated_cost_usd      DOUBLE,
  inserted_at             TIMESTAMP,
  synced_to_zgrc          BOOLEAN DEFAULT FALSE,
  synced_at               TIMESTAMP
)
USING DELTA
PARTITIONED BY (query_date)
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.feature.allowColumnDefaults' = 'supported'
);

In [0]:
%sql
-- Step 1: Check if there's new data in the last 5 minutes
CREATE OR REPLACE TEMPORARY VIEW new_data_check AS
SELECT COUNT(*) as new_record_count
FROM system.query.history
WHERE query_source.genie_space_id IS NOT NULL
  AND start_time >= CURRENT_TIMESTAMP() - INTERVAL 30 MINUTES;

-- Step 2: Only proceed if new data exists
SELECT * FROM new_data_check;

new_record_count
17


In [ ]:
%python
# Check if we have new data
new_data = spark.sql("SELECT new_record_count FROM new_data_check").first()[0]

# if new_data > 0:
if True:
    print(f"Found {new_data} new Genie queries in last 5 minutes. Running cost calculation...")
    
    # Run your MERGE query
    spark.sql("""
    MERGE INTO zeb_labs.default.genie_cost_daily AS target
    USING (
      WITH warehouse_info AS (
        SELECT
          warehouse_id,
          warehouse_name,
          warehouse_size,
          CASE warehouse_size
            WHEN '2X_SMALL' THEN 1
            WHEN 'X_SMALL'  THEN 2
            WHEN 'SMALL'    THEN 4
            WHEN 'MEDIUM'   THEN 8
            WHEN 'LARGE'    THEN 16
            WHEN 'X_LARGE'  THEN 32
            WHEN '2X_LARGE' THEN 64
            WHEN '3X_LARGE' THEN 128
            WHEN '4X_LARGE' THEN 256
            ELSE 1
          END AS dbu_per_hour
        FROM system.compute.warehouses
        WHERE warehouse_id = <WAREHOUSE_ID>
      ),
      
      genie_queries AS (
        SELECT
          statement_id,
          executed_by,
          query_source.genie_space_id AS genie_space_id,
          compute.warehouse_id AS warehouse_id,
          execution_status,
          start_time,
          end_time,
          CASE
            WHEN execution_status = 'FAILED' AND execution_duration_ms IS NULL THEN 0
            ELSE COALESCE(execution_duration_ms, total_duration_ms, 0)
          END AS duration_ms,
          DATE(start_time) AS query_date
        FROM system.query.history
        WHERE query_source.genie_space_id IS NOT NULL
          AND start_time >= TRUNC(CURRENT_DATE(), 'MM')
          AND start_time <= CURRENT_TIMESTAMP()
      ),
      
      dbu_estimate AS (
        SELECT
          g.statement_id,
          g.executed_by,
          g.genie_space_id,
          g.warehouse_id,
          g.execution_status,
          g.start_time,
          g.end_time,
          g.duration_ms,
          g.query_date,
          w.warehouse_size,
          w.dbu_per_hour,
          ROUND((g.duration_ms / 3600000.0) * w.dbu_per_hour, 8) AS estimated_dbus
        FROM genie_queries g
        JOIN warehouse_info w ON g.warehouse_id = w.warehouse_id
      ),
      
      prices AS (
        SELECT
          sku_name,
          CAST(pricing.default AS DOUBLE) AS price_per_dbu
        FROM system.billing.list_prices
        WHERE sku_name = 'ENTERPRISE_SERVERLESS_SQL_COMPUTE'
          AND price_end_time IS NULL
        LIMIT 1
      )
      
      SELECT
        d.query_date,
        d.executed_by,
        d.genie_space_id,
        d.warehouse_id,
        d.warehouse_size,
        d.dbu_per_hour,
        d.execution_status,
        COUNT(*) AS query_count,
        SUM(d.duration_ms) AS total_duration_ms,
        ROUND(SUM(d.estimated_dbus), 8) AS total_estimated_dbus,
        p.price_per_dbu,
        ROUND(SUM(d.estimated_dbus) * p.price_per_dbu, 6) AS estimated_cost_usd,
        CURRENT_TIMESTAMP() AS inserted_at,
        FALSE AS synced_to_zgrc,
        CAST(NULL AS TIMESTAMP) AS synced_at
      FROM dbu_estimate d
      CROSS JOIN prices p
      GROUP BY
        d.query_date, d.executed_by, d.genie_space_id, d.warehouse_id,
        d.warehouse_size, d.dbu_per_hour, d.execution_status, p.price_per_dbu
    ) AS source
    
    ON target.query_date = source.query_date
      AND target.executed_by = source.executed_by
      AND target.genie_space_id = source.genie_space_id
      AND target.execution_status = source.execution_status
    
    WHEN MATCHED THEN
      UPDATE SET
        target.warehouse_id = source.warehouse_id,
        target.warehouse_size = source.warehouse_size,
        target.dbu_per_hour = source.dbu_per_hour,
        target.query_count = source.query_count,
        target.total_duration_ms = source.total_duration_ms,
        target.total_estimated_dbus = source.total_estimated_dbus,
        target.price_per_dbu = source.price_per_dbu,
        target.estimated_cost_usd = source.estimated_cost_usd,
        target.inserted_at = source.inserted_at,
        target.synced_to_zgrc = FALSE
    
    WHEN NOT MATCHED THEN INSERT *
    """)
    
    # Show results
    result = spark.sql("""
        SELECT
          COUNT(*) AS total_rows,
          MIN(query_date) AS earliest_date,
          MAX(query_date) AS latest_date,
          MAX(inserted_at) AS last_run_at,
          ROUND(SUM(estimated_cost_usd), 4) AS total_cost_usd_in_table
        FROM zeb_labs.default.genie_cost_daily
    """)
    
    display(result)
    print("Cost calculation completed!")
    
else:
    print(f"No new Genie queries in last 5 minutes. Skipping calculation.")

Passing usage cost to ZGRC application

In [0]:
import asyncio
import logging
from dataclasses import dataclass
from typing import Dict, List, Optional, Any

import requests
import httpx
from pydantic import BaseModel
from tenacity import retry, stop_after_attempt, wait_exponential

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [ ]:
import os
from dataclasses import field


def _required_env(var_name: str, placeholder: str) -> str:
    """Read a config value from the environment, falling back to a placeholder.

    On Databricks prefer secret scopes, e.g.:
        dbutils.secrets.get(scope="genie-governance", key="databricks_token")
    """
    return os.environ.get(var_name, placeholder)


@dataclass
class Config:
    """System configuration.

    Values are read from environment variables so no secrets are committed.
    Set these before running (or swap _required_env for dbutils.secrets.get on Databricks):
      DATABRICKS_HOST, DATABRICKS_TOKEN, GENIE_SPACE_ID,
      ZGRC_BASE_URL, ZGRC_AUTH_TOKEN, ZGRC_GROUP_ID, ZGRC_SERVICE_KEY, GENIE_COST_TABLE
    """

    # Databricks workspace URL, e.g. "https://dbc-xxxxxxxx-xxxx.cloud.databricks.com"
    databricks_host: str = field(default_factory=lambda: _required_env("DATABRICKS_HOST", "<YOUR_DATABRICKS_HOST>"))
    # Databricks personal access token ("dapi..."). Never commit; use a secret scope.
    databricks_token: str = field(default_factory=lambda: _required_env("DATABRICKS_TOKEN", "<YOUR_DATABRICKS_TOKEN>"))
    # Databricks Genie space id being governed.
    genie_space_id: str = field(default_factory=lambda: _required_env("GENIE_SPACE_ID", "<YOUR_GENIE_SPACE_ID>"))

    # Base URL of the Z-GRC application.
    zgrc_base_url: str = field(default_factory=lambda: _required_env("ZGRC_BASE_URL", "https://z-grc.zeb.co"))
    # Z-GRC auth token (JWT) used as the auth_token cookie. Short-lived; rotate before expiry.
    zgrc_auth_token: str = field(default_factory=lambda: _required_env("ZGRC_AUTH_TOKEN", "<YOUR_ZGRC_AUTH_TOKEN>"))
    # Z-GRC group/policy id that owns the users and quotas.
    zgrc_group_id: str = field(default_factory=lambda: _required_env("ZGRC_GROUP_ID", "<YOUR_ZGRC_GROUP_ID>"))
    # Z-GRC service key ("sk_...") authorizing external API-key/user creation.
    zgrc_service_key: str = field(default_factory=lambda: _required_env("ZGRC_SERVICE_KEY", "<YOUR_ZGRC_SERVICE_KEY>"))

    # Fully-qualified Delta table holding the computed Genie costs.
    cost_table: str = field(default_factory=lambda: _required_env("GENIE_COST_TABLE", "zeb_labs.default.genie_cost_daily"))

    def validate(self):
        """Validate configuration"""
        required = [
            self.databricks_host,
            self.databricks_token,
            self.genie_space_id,
            self.zgrc_base_url,
            self.zgrc_auth_token,
            self.zgrc_group_id,
            self.zgrc_service_key,
            self.cost_table
        ]
        if not all(required):
            raise ValueError("Missing required configuration")
        # Guard against running with the committed placeholders still in place.
        if any(str(v).startswith("<YOUR_") for v in required):
            raise ValueError("Configuration still contains placeholders - set the required environment variables")

config = Config()
config.validate()


In [0]:
class APIClient:
    """HTTP client with retry logic"""

    def __init__(self, base_url: str, auth_token: str = None, timeout: int = 60):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.auth_token = auth_token

    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=1, max=10))
    async def get(self, endpoint: str, params: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        url = f"{self.base_url}/{endpoint.lstrip('/')}"
        headers = {}
        if self.auth_token:
            headers["Cookie"] = f"auth_token={self.auth_token}"

        async with httpx.AsyncClient(timeout=self.timeout, trust_env=False) as client:
            response = await client.get(url, params=params, headers=headers)
            response.raise_for_status()
            return response.json()

    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=1, max=10))
    async def post(
        self,
        endpoint: str,
        json: Optional[Dict[str, Any]] = None,
        headers: Optional[Dict[str, str]] = None
    ) -> Dict[str, Any]:
        url = f"{self.base_url}/{endpoint.lstrip('/')}"
        request_headers = {}
        if self.auth_token:
            request_headers["Cookie"] = f"auth_token={self.auth_token}"
        if headers:
            request_headers.update(headers)

        async with httpx.AsyncClient(timeout=self.timeout, trust_env=False) as client:
            response = await client.post(url, json=json, headers=request_headers)
            response.raise_for_status()
            return response.json()
        
class Quota(BaseModel):
    used_quota: float = 0.0
    remaining_quota: float = 0.0


class ZGRCClient:

    def __init__(self, cfg: Config):
        self.config = cfg
        self.client = APIClient(
            base_url=cfg.zgrc_base_url,
            auth_token=cfg.zgrc_auth_token
        )
        self._user_cache: Dict[str, str] = {}

    async def get_user_id(self, email: str) -> Optional[str]:
        """Get user_id from email with caching"""
        if email in self._user_cache:
            return self._user_cache[email]

        try:
            response = await self.client.get(f"/api/groups/{self.config.zgrc_group_id}")
            for member in response.get('group', {}).get('members', []):
                user = member.get('user', {})
                user_email = user.get('email')
                user_id = user.get('user_id')
                if user_email and user_id:
                    self._user_cache[user_email] = user_id

            return self._user_cache.get(email)
        except httpx.HTTPStatusError as e:
            if e.response.status_code == 401:
                logger.error("Authentication failed. ZGRC auth token may be expired.")
            else:
                logger.error(f"Failed to fetch user mapping: HTTP {e.response.status_code}")
            return None
        except Exception as e:
            logger.error(f"Failed to fetch user mapping: {e}")
            return None

    async def add_user(self, email: str) -> Optional[str]:
        """Add a new user to ZGRC group"""
        payload = {
            "email": email,
            "group_id": self.config.zgrc_group_id,
            "name": email.split('@')[0],
            "description": "Auto-added by Genie Cost Governance"
        }

        headers = {"X-Service-Key": self.config.zgrc_service_key}

        try:
            logger.info(f"Adding user to ZGRC: {email}")
            response = await self.client.post(
                "/api/external/apikey/generate",
                json=payload,
                headers=headers
            )

            key_data = response.get('key', {})
            user_id = key_data.get('user_id')

            if user_id:
                self._user_cache[email] = user_id
                logger.info(f"User added successfully: {user_id}")
                logger.warning(f"IMPORTANT: Set quota for {email} in ZGRC UI at {self.config.zgrc_base_url}")

            return user_id
        except httpx.HTTPStatusError as e:
            logger.error(f"Failed to add user {email}: HTTP {e.response.status_code} - {e.response.text}")
            return None
        except Exception as e:
            logger.error(f"Failed to add user {email}: {e}")
            return None

    async def get_quota(self, user_id: str) -> Optional[Quota]:
        """Get current quota for a user"""
        try:
            params = {
                "group_id": self.config.zgrc_group_id,
                "user_id": user_id,
            }
            response = await self.client.get("/api/quota/user", params=params)
            return Quota(
                used_quota=response.get("used_cost", 0.0),
                remaining_quota=response.get("remaining_cost", 0.0),
            )
        except httpx.HTTPStatusError as e:
            if e.response.status_code == 404:
                logger.error(f"User {user_id} not found in ZGRC")
            else:
                logger.error(f"Failed to get quota for user {user_id}: HTTP {e.response.status_code}")
            return None
        except Exception as e:
            logger.error(f"Failed to get quota for user {user_id}: {e}")
            return None

    async def post_usage(self, user_id: str, tokens_used: int, cost: float) -> Optional[Quota]:
        """Post usage to ZGRC"""
        body = {
            "user_id": user_id,
            "policy_id": self.config.zgrc_group_id,
            "group_id": self.config.zgrc_group_id,
            "amount": tokens_used,
            "cost": cost,
        }

        try:
            logger.info(f"Posting usage for user {user_id}: ${cost:.4f}")
            response = await self.client.post("/api/quota/consume", json=body)
            return Quota(
                used_quota=response.get("used_cost", 0.0),
                remaining_quota=response.get("remaining_cost", 0.0),
            )
        except httpx.HTTPStatusError as e:
            if e.response.status_code == 403:
                logger.error(f"User {user_id} has no quota allocated. Please set budget in ZGRC UI: {self.config.zgrc_base_url}")
            elif e.response.status_code == 404:
                logger.error(f"User {user_id} not found in ZGRC group")
            else:
                logger.error(f"Failed to post usage for user {user_id}: HTTP {e.response.status_code} - {e.response.text}")
            return None
        except Exception as e:
            logger.error(f"Failed to post usage for user {user_id}: {e}")
            return None


class GenieAccessManager:

    def __init__(self, cfg: Config):
        self.config = cfg
        self.headers = {
            "Authorization": f"Bearer {cfg.databricks_token}",
            "Content-Type": "application/json"
        }

    def _get(self, path: str, params: Optional[dict] = None) -> requests.Response:
        return requests.get(
            f"{self.config.databricks_host}{path}",
            headers=self.headers,
            params=params,
            timeout=30
        )

    def _patch(self, path: str, payload: dict) -> requests.Response:
        return requests.patch(
            f"{self.config.databricks_host}{path}",
            headers=self.headers,
            json=payload,
            timeout=30
        )

    def _get_permissions(self) -> Optional[dict]:
        """Fetch current permission structure"""
        resource_name = f"datarooms/{self.config.genie_space_id}"
        path = f"/api/2.0/accesspolicies?name={resource_name}"

        try:
            response = self._get(path)
            if response.status_code == 200:
                data = response.json()
                policy = data.get("access_policy", data)
                return {
                    "name": policy.get("name", resource_name),
                    "internal_name": policy.get("internal_name", ""),
                    "permissions": policy.get("permissions", []),
                }
        except Exception as e:
            logger.error(f"Failed to get permissions: {e}")
        return None

    def _lookup_user_principal(self, user_email: str) -> Optional[str]:
        """Get Databricks principal ID for a user"""
        path = "/api/2.0/preview/scim/v2/Users"
        params = {"filter": f'userName eq "{user_email}"'}

        try:
            response = self._get(path, params=params)
            if response.status_code == 200:
                data = response.json()
                if data.get("totalResults", 0) > 0:
                    user_id = data["Resources"][0].get("id")
                    return f"principals/{user_id}"
        except Exception as e:
            logger.error(f"Failed to lookup user principal for {user_email}: {e}")
        return None

    def grant_access(self, user_email: str, permission_level: str = "CAN_RUN") -> bool:
        """Grant Genie access to a user"""
        principal = self._lookup_user_principal(user_email)
        if not principal:
            logger.error(f"User not found in Databricks: {user_email}")
            return False

        perm_data = self._get_permissions()
        if not perm_data:
            logger.error("Could not retrieve permissions")
            return False

        payload = {
            "access_policy": {
                "name": perm_data.get("name"),
                "permissions": [{
                    "principal": principal,
                    "permissions": [permission_level]
                }]
            },
            "send_notification": False
        }

        if internal_name := perm_data.get("internal_name"):
            payload["access_policy"]["internal_name"] = internal_name

        try:
            response = self._patch("/api/2.0/accesspolicies", payload)
            if response.status_code == 200:
                logger.info(f"Access granted: {user_email}")
                return True
            logger.error(f"Grant failed: HTTP {response.status_code}")
            return False
        except Exception as e:
            logger.error(f"Grant failed for {user_email}: {e}")
            return False

    def revoke_access(self, user_email: str) -> bool:
        """Revoke Genie access from a user"""
        principal = self._lookup_user_principal(user_email)
        if not principal:
            logger.error(f"User not found in Databricks: {user_email}")
            return False

        perm_data = self._get_permissions()
        if not perm_data:
            logger.error("Could not retrieve permissions")
            return False

        payload = {
            "access_policy": {
                "name": perm_data.get("name"),
                "permissions": [{
                    "principal": principal,
                    "permissions": []
                }]
            },
            "send_notification": False
        }

        if internal_name := perm_data.get("internal_name"):
            payload["access_policy"]["internal_name"] = internal_name

        try:
            response = self._patch("/api/2.0/accesspolicies", payload)
            if response.status_code == 200:
                logger.info(f"Access revoked: {user_email}")
                return True
            logger.error(f"Revoke failed: HTTP {response.status_code}")
            return False
        except Exception as e:
            logger.error(f"Revoke failed for {user_email}: {e}")
            return False

class DataLayer:
    """Handle database operations"""

    def __init__(self, cfg: Config):
        self.config = cfg

    def get_unsynced_user_costs(self) -> List[Dict]:
        """Get costs for records not yet synced to ZGRC"""
        try:
            df = spark.sql(f"""
                SELECT
                    executed_by as user_email,
                    SUM(estimated_cost_usd) as new_cost,
                    COUNT(*) as unsynced_records
                FROM {self.config.cost_table}
                WHERE synced_to_zgrc = FALSE
                GROUP BY executed_by
                HAVING SUM(estimated_cost_usd) > 0
                ORDER BY new_cost DESC
            """)
            return [row.asDict() for row in df.collect()]
        except Exception as e:
            logger.error(f"Failed to get unsynced costs: {e}")
            return []

    def mark_as_synced(self, user_email: str) -> bool:
        """Mark user's records as synced to ZGRC"""
        try:
            spark.sql(f"""
                UPDATE {self.config.cost_table}
                SET
                    synced_to_zgrc = TRUE,
                    synced_at = CURRENT_TIMESTAMP()
                WHERE executed_by = '{user_email}'
                  AND synced_to_zgrc = FALSE
            """)
            logger.info(f"Marked records as synced: {user_email}")
            return True
        except Exception as e:
            logger.error(f"Failed to mark as synced for {user_email}: {e}")
            return False

    def mark_as_failed(self, user_email: str, error_message: str) -> bool:
        """Mark user's records with error for manual review"""
        try:
            spark.sql(f"""
                UPDATE {self.config.cost_table}
                SET
                    synced_to_zgrc = FALSE
                WHERE executed_by = '{user_email}'
                  AND synced_to_zgrc = FALSE
            """)
            logger.info(f"Kept records as unsynced for manual review: {user_email}")
            return True
        except Exception as e:
            logger.error(f"Failed to update error status for {user_email}: {e}")
            return False



In [0]:
class GenieGovernanceOrchestrator:
    """Main orchestration logic"""

    def __init__(self, cfg: Config):
        self.config = cfg
        self.zgrc = ZGRCClient(cfg)
        self.genie = GenieAccessManager(cfg)
        self.data = DataLayer(cfg)
        self.results = {
            "processed": 0,
            "success": 0,
            "failed": 0,
            "blocked": 0,
            "needs_quota": 0,
            "newly_created": 0
        }

    async def ensure_user_exists(self, user_email: str) -> tuple[Optional[str], bool]:
        """Ensure user exists in ZGRC, create if not. Returns (user_id, was_created)"""
        user_id = await self.zgrc.get_user_id(user_email)
        if user_id:
            return user_id, False
        
        user_id = await self.zgrc.add_user(user_email)
        if user_id:
            return user_id, True
        
        return None, False

    async def check_user_has_quota(self, user_id: str, user_email: str) -> bool:
        """Check if user has quota allocated"""
        quota = await self.zgrc.get_quota(user_id)
        if not quota:
            logger.warning(f"Cannot retrieve quota for {user_email}")
            return False
        
        total_quota = quota.used_quota + quota.remaining_quota
        if total_quota == 0:
            logger.warning(f"User {user_email} has no quota allocated (total budget = $0)")
            return False
        
        return True

    async def process_user(self, user_email: str, new_cost: float) -> bool:
        """Process a single user's cost governance"""
        logger.info(f"Processing user: {user_email}, New cost: ${new_cost:.4f}")

        try:
            user_id, was_created = await self.ensure_user_exists(user_email)
            
            if not user_id:
                logger.error(f"Cannot proceed without user_id for {user_email}")
                self.data.mark_as_failed(user_email, "Failed to get/create user_id")
                self.results["failed"] += 1
                return False

            if was_created:
                logger.info(f"User {user_email} was just created in ZGRC")
                self.results["newly_created"] += 1
                
                has_quota = await self.check_user_has_quota(user_id, user_email)
                if not has_quota:
                    logger.warning(f"Newly created user {user_email} has no quota. Skipping for now - will retry next run after quota is allocated.")
                    self.data.mark_as_failed(user_email, "New user - waiting for quota allocation")
                    self.results["needs_quota"] += 1
                    self.results["failed"] += 1
                    return False

            estimated_tokens = int(new_cost * 1000000)
            updated_quota = await self.zgrc.post_usage(
                user_id=user_id,
                tokens_used=estimated_tokens,
                cost=new_cost
            )

            if not updated_quota:
                logger.error(f"Failed to post usage for {user_email}. User may need quota allocation in ZGRC UI.")
                self.data.mark_as_failed(user_email, "No quota allocated - needs manual setup")
                self.results["needs_quota"] += 1
                self.results["failed"] += 1
                return False

            logger.info(f"User {user_email} - Used: ${updated_quota.used_quota:.4f}, Remaining: ${updated_quota.remaining_quota:.4f}")

            if updated_quota.remaining_quota <= 0:
                logger.warning(f"Quota exceeded for {user_email}, revoking access")
                if self.genie.revoke_access(user_email):
                    self.results["blocked"] += 1
                else:
                    logger.error(f"Failed to revoke access for {user_email}")

            if self.data.mark_as_synced(user_email):
                self.results["success"] += 1
                return True
            else:
                self.results["failed"] += 1
                return False

        except Exception as e:
            logger.error(f"Unexpected error processing user {user_email}: {e}")
            self.data.mark_as_failed(user_email, str(e))
            self.results["failed"] += 1
            return False

    async def run(self):
        """Main execution flow"""
        logger.info("="*60)
        logger.info("Starting Genie Cost Governance")
        logger.info("="*60)

        user_costs = self.data.get_unsynced_user_costs()
        self.results["processed"] = len(user_costs)

        if not user_costs:
            logger.info("No unsynced records found")
            return self.results

        logger.info(f"Found {len(user_costs)} users with unsynced usage")

        for user_data in user_costs:
            user_email = user_data['user_email']
            new_cost = user_data['new_cost']
            await self.process_user(user_email, new_cost)

        logger.info("="*60)
        logger.info("Execution Summary:")
        logger.info(f"  Processed: {self.results['processed']}")
        logger.info(f"  Success: {self.results['success']}")
        logger.info(f"  Failed: {self.results['failed']}")
        logger.info(f"  Blocked: {self.results['blocked']}")
        logger.info(f"  Newly Created: {self.results['newly_created']}")
        logger.info(f"  Needs Quota Setup: {self.results['needs_quota']}")
        logger.info("="*60)

        if self.results['needs_quota'] > 0:
            logger.warning(f"ACTION REQUIRED: {self.results['needs_quota']} users need quota allocation in ZGRC UI")
            logger.warning(f"Visit: {self.config.zgrc_base_url}")
            logger.warning("After allocating quota, re-run this job to sync their usage.")

        return self.results

In [0]:
# if new_data > 0:
if True:
    orchestrator = GenieGovernanceOrchestrator(config)
    results = await orchestrator.run()
    print(results)

2026-05-22 16:34:46,989 - __main__ - INFO - ============================================================
2026-05-22 16:34:46,990 - __main__ - INFO - Starting Genie Cost Governance
2026-05-22 16:34:46,990 - __main__ - INFO - ============================================================
2026-05-22 16:34:47,626 - __main__ - INFO - Found 4 users with unsynced usage
2026-05-22 16:34:47,627 - __main__ - INFO - Processing user: shriyokesh.thangavel@zeb.co, New cost: $0.0344
2026-05-22 16:34:47,888 - httpx - INFO - HTTP Request: GET https://z-grc.zeb.co/api/groups/019e5083-e1bc-75bd-9466-c0edb30e31ba "HTTP/1.1 401 Unauthorized"
2026-05-22 16:34:48,995 - httpx - INFO - HTTP Request: GET https://z-grc.zeb.co/api/groups/019e5083-e1bc-75bd-9466-c0edb30e31ba "HTTP/1.1 401 Unauthorized"
2026-05-22 16:34:51,094 - httpx - INFO - HTTP Request: GET https://z-grc.zeb.co/api/groups/019e5083-e1bc-75bd-9466-c0edb30e31ba "HTTP/1.1 401 Unauthorized"
2026-05-22 16:34:51,096 - __main__ - ERROR - Failed to fetch 

{'processed': 4, 'success': 4, 'failed': 0, 'blocked': 0, 'needs_quota': 0, 'newly_created': 4}
